# 금융/증권 자동 포스팅 전체 파이프라인

스크래핑 → LLM 요약 → 티스토리 발행을 한 번에 실행하는 **메인 노트북** 입니다.

In [ ]:
from app.core.utils.notebook import run_async
from app.modules.browser.playwright import PlaywrightManager
from app.modules.llm.groq import create_async_groq_client
from app.prompts.finance_news import FINANCE_NEWS_SYSTEM_PROMPT
from app.schemas.enums import BlogCategory
from app.schemas.tistory.article import ReservationData
from app.schemas.tistory.request import TistoryPublishRequest
from app.services.llm.news_summarize import NewsSummarizeService
from app.services.scraper.finance_news import FinanceNewsScrapService
from app.services.tistory.post import TistoryPostService

In [ ]:
fns = FinanceNewsScrapService(PlaywrightManager())
articles = run_async(fns.do_scraping())

ns = NewsSummarizeService(create_async_groq_client())
summarized_articles = run_async(ns.summarize_many(articles, FINANCE_NEWS_SYSTEM_PROMPT))

tistory_post_service = TistoryPostService(PlaywrightManager())

# 예약 지정
""" reservation_data = ReservationData(
    type='fix',
    date='2026-06-10',
    time='09:45'
) """
# 예약 랜덤
reservation_data = ReservationData(
    type='random',
    date='2026-06-23'
)

payload = TistoryPublishRequest(
    blog_category=BlogCategory.finance,
    summarized_articles=summarized_articles,
    reservation_data=reservation_data
)

response = run_async(tistory_post_service.do_posting(payload.blog_category, payload.summarized_articles, payload.reservation_data))

print('DONE')